# m08 lab · Action potentials and ionic models

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

The cardiac membrane is a capacitor with ion channels in parallel. You start with the passive circuit and Nernst
potentials, integrate a gating variable, explore threshold and firing in the two-variable FitzHugh–Nagumo model, and
then measure restitution in the Mitchell–Schaeffer model of the cardiac action potential.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The membrane as a circuit; Nernst potentials | eq-m08-membrane, eq-m08-nernst |
| B | Gates under voltage clamp; exact and Euler updates | eq-m08-gating |
| C | FitzHugh–Nagumo: threshold, refractoriness, repetitive firing | eq-m08-fhn, eq-m08-hopf |
| D | Mitchell–Schaeffer: the action potential and S1–S2 restitution | eq-m08-ms, eq-m08-apd |
| E | Shorter action potentials, as in electrical remodeling | eq-m08-apd |

Mitchell–Schaeffer parameters are the values Nieto Ramos et al. (2023) report as based on the original paper;
FitzHugh–Nagumo parameters, concentrations and stimulus settings are illustrative. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import membrane as M
P = M.MS_PARAMS
print(P)

## Part A · The membrane as a circuit

With no active channels the membrane is a resistor and a capacitor in parallel (eq-m08-membrane): a current step
charges it with time constant $\tau_m = RC$. Each ion species pulls the voltage toward its Nernst potential
(eq-m08-nernst); $RT/F$ comes from the exact SI constants.

In [ ]:
t = np.linspace(0, 50, 501)
v = M.passive_step(t, current=1.0, resistance=10.0, capacitance=1.0)    # illustrative units: tau = 10
tau_meas = t[np.argmax(v >= (1 - np.exp(-1)) * 10.0)]
print(f"RT/F at 37 C: {M.thermal_voltage():.2f} mV; per tenfold gradient: {M.thermal_voltage() * np.log(10):.1f} mV")
for name, co, ci, z in (("30-fold gradient, z = +1", 5.0, 150.0, 1), ("10-fold gradient, z = +2", 2.0, 0.2, 2)):
    print(f"{name} (illustrative concentrations): E = {M.nernst(co, ci, z):.1f} mV")
print(f"time to 63% of the final voltage: {tau_meas:.1f} (RC = 10)")
plt.plot(t, v); plt.xlabel("time (RC units)"); plt.ylabel("V"); plt.show()

In [ ]:
# Checks
assert abs(tau_meas - 10.0) < 0.15
assert abs(M.thermal_voltage() * np.log(10) - 61.54) < 0.01

## Part B · Gates under voltage clamp

A Hodgkin–Huxley gate opens at rate $\alpha$ and closes at rate $\beta$ (eq-m08-gating). At fixed voltage the rates
are constant, the gate relaxes exponentially, and the update $m \leftarrow m_\infty + (m - m_\infty)e^{-\Delta t/\tau}$
is exact for any step. Forward Euler is not: compare the two at large steps. Simulators use the exponential update
for gates for this reason.

In [ ]:
alpha, beta, m0 = 0.4, 0.1, 0.0                          # illustrative rates (1/ms)
exact = M.gate_clamp(4.0, m0, alpha, beta)                # tau = 1 / (alpha + beta) = 2 ms
for dt in (0.01, 0.5, 1.0, 2.0):
    m_exp = m0
    m_inf, tau = M.gate_steady(alpha, beta)
    for _ in range(int(round(4.0 / dt))):
        m_exp = m_inf + (m_exp - m_inf) * np.exp(-dt / tau)
    print(f"dt = {dt:5.2f} ms: exponential update error {abs(m_exp - exact):.1e}, "
          f"Euler error {abs(M.euler_gate(4.0, dt, m0, alpha, beta) - exact):.1e}")
err_exp, err_euler = abs(m_exp - exact), abs(M.euler_gate(4.0, 2.0, m0, alpha, beta) - exact)

In [ ]:
# Checks
assert err_exp < 1e-12 and err_euler > 1e-2

## Part C · FitzHugh–Nagumo: threshold and firing

Two variables are enough for excitability (eq-m08-fhn): a fast voltage-like $v$ with a cubic nullcline and a slow
recovery variable $w$. A kick past the middle branch of the cubic sends the state around a long loop, the action
potential. A steady current moves the rest state; when it reaches the point where the Jacobian's trace vanishes the
cell fires on its own (eq-m08-hopf).

In [ ]:
th = M.fhn_threshold()
IH = M.fhn_hopf_current()
peaks = {k: M.fhn_simulate(60, 0.02, kick=k * th)[1].max() for k in (0.9, 0.99, 1.01, 1.1)}
for k, pk in peaks.items():
    print(f"kick {k:.2f} x threshold: peak v = {pk:.2f}")
t, v1, w1 = M.fhn_simulate(60, 0.02, kick=1.1 * th)
i = np.argmax((t > 5) & (v1 < 0))                          # recovering: v back below 0, w still high
_, v2, _ = M.fhn_simulate(40, 0.02, v0=v1[i], w0=w1[i], kick=1.1 * th)
print(f"a second kick at t = {t[i]:.1f}, during recovery, reaches v = {v2.max():.2f}")
amps = {}
for I in (IH - 0.03, IH + 0.03):
    tt, vv, _ = M.fhn_simulate(400, 0.02, current=I, kick=0.01)
    amps[I] = np.ptp(vv[tt > 250])
    print(f"I = {I:.3f} (Hopf value {IH:.3f}): late peak-to-peak v = {amps[I]:.2f}")
vs = np.linspace(-2.5, 2.5, 200)
plt.figure(figsize=(4.5, 3.5))
plt.plot(vs, vs - vs**3 / 3, "gray"); plt.plot(vs, (vs + 0.7) / 0.8, "gray", ls="--"); plt.plot(v1, w1)
plt.xlim(-2.5, 2.5); plt.ylim(-1, 2); plt.xlabel("v"); plt.ylabel("w"); plt.show()

In [ ]:
# Checks
assert peaks[0.99] < 0.5 and peaks[1.01] > 1.5           # all or none
assert v2.max() < v1.max()                                 # relative refractoriness
assert amps[IH - 0.03] < 0.1 and amps[IH + 0.03] > 1.0

## Part D · Mitchell–Schaeffer: the action potential and restitution

The Mitchell–Schaeffer model (eq-m08-ms) has an inward current gated by $h$ and a passive outward current. The excited
state exists only while $h \ge h_{\min} = 4\tau_{\text{in}}/\tau_{\text{out}}$; during the action potential $h$ decays
with time constant $\tau_{\text{close}}$, so the excited state disappears after $\tau_{\text{close}}\ln(h_0/h_{\min})$
(eq-m08-apd). Measure it, then run the S1–S2 protocol: four beats 500 ms apart and one premature beat.

In [ ]:
t, v, h = M.ms_cell(600, [0.0], P)
up = np.argmax(v > P["v_gate"])
t_hmin = t[np.argmax(h < M.ms_h_min(**P))] - t[up]
apd = M.apds(t, v, P["v_gate"])[0, 1]
print(f"h_min = {M.ms_h_min(**P):.2f}; excited state disappears at {t_hmin:.1f} ms "
      f"(closed form {M.ms_apd_closed(np.inf, **P):.1f} ms); v reaches v_gate at {apd:.1f} ms")
s2s = np.array([225, 230, 240, 260, 280, 300, 350, 400, 450, 500, 600, 800])
res = M.s1s2(s2s, P)
ok = np.isfinite(res[:, 2])
lo, hi = 150.0, 230.0                                      # refractory period by bisection on the S2 interval
for _ in range(12):
    mid = (lo + hi) / 2
    if np.isfinite(M.s1s2([mid], P)[0, 2]):
        hi = mid
    else:
        lo = mid
erp = hi
print(f"shortest S1-S2 interval that captures (effective refractory period): {erp:.1f} ms")
dis = np.linspace(0, 600, 300)
plt.plot(dis, M.ms_apd_closed(dis, **P), "gray", label="closed form")
plt.plot(res[ok, 1], res[ok, 2], "o", label="simulated S2 beats")
plt.xlabel("diastolic interval (ms)"); plt.ylabel("APD (ms)"); plt.legend(); plt.show()
gap = res[ok, 2] - M.ms_apd_closed(res[ok, 1], **P)
print(f"simulated APD minus closed form, for DI >= 40 ms: {gap[res[ok, 1] >= 40].min():.0f} to {gap[res[ok, 1] >= 40].max():.0f} ms")
print(f"closed-form restitution slope at DI -> 0: {M.ms_restitution_slope(0.0, **P):.1f}")

The closed form predicts when the excited state disappears, and it does so to within a fraction of a millisecond.
The voltage then takes another 25–40 ms to fall through $v_{\text{gate}}$, because it lingers where the excited state
used to be. So the closed form gives the shape of restitution, not the measured duration. The slope is steep at short
diastolic intervals; m10 shows why a slope above 1 matters.

In [ ]:
# Checks
assert abs(t_hmin - M.ms_apd_closed(np.inf, **P)) < 1.0
assert np.all((gap[res[ok, 1] >= 40] > 25) & (gap[res[ok, 1] >= 40] < 45))
assert np.all(np.diff(res[ok, 2]) > 0)                    # restitution rises with the interval
assert 210 < erp < 225

## Part E · Shorter action potentials

In goats kept in atrial fibrillation, the atrial effective refractory period shortened by 35% and recovered within a week
of sinus rhythm (Wijffels et al. 1995). In this model the action potential scales with
$\tau_{\text{close}}$, so shortening it by 35% is a crude stand-in for that change. It is an illustration, not a model of
the ionic remodeling itself. Compare restitution and the refractory period.

In [ ]:
Q = dict(P, tau_close=0.65 * P["tau_close"])
res2 = M.s1s2(s2s, Q)
ok2 = np.isfinite(res2[:, 2])
lo, hi = 100.0, 230.0
for _ in range(12):
    mid = (lo + hi) / 2
    if np.isfinite(M.s1s2([mid], Q)[0, 2]):
        hi = mid
    else:
        lo = mid
erp2 = hi
apd_long = (res[res[:, 0] == 800, 2][0], res2[res2[:, 0] == 800, 2][0])
print(f"APD at S2 = 800 ms: {apd_long[0]:.0f} ms -> {apd_long[1]:.0f} ms; refractory period {erp:.0f} -> {erp2:.0f} ms")
plt.plot(res[ok, 1], res[ok, 2], "o-", label="tau_close = 120 ms")
plt.plot(res2[ok2, 1], res2[ok2, 2], "o-", label="tau_close = 78 ms")
plt.xlabel("diastolic interval (ms)"); plt.ylabel("APD (ms)"); plt.legend(); plt.show()

In [ ]:
# Checks
assert apd_long[1] < 0.75 * apd_long[0]
assert erp2 < erp - 40

## Where to go next

- Replace the Heaviside switch of $h$ at $v_{\text{gate}}$ with a smooth $\tanh$ and see how restitution changes.
- Pace the cell at shorter and shorter cycle lengths and watch the action potential durations alternate; m10 explains
  when that happens.
- Couple many cells along a line with diffusion: that is m09.